## AI for Sustainability – SRIP 2026
## Land Use Classification

##Author: Ananya Roy

##Dataset: Delhi-NCR Sentinel-2 & ESA WorldCover 2021

#Q1. Spatial Reasoning & Data Filtering

##a. Plot the Delhi-NCR shapefile using matplotlib and overlay a 60×60 km uniform grid




In [ ]:
!pip install geopandas shapely --quiet
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
import matplotlib.pyplot as plt

# Load the shapefile
ncr = gpd.read_file('/content/drive/MyDrive/AI_Sustainability/Delhi/delhi_ncr_region.geojson')

# Project to EPSG:32644 (UTM zone 44N)
ncr_metric = ncr.to_crs(epsg=32644)

# Create a 60x60 km grid
xmin, ymin, xmax, ymax = ncr_metric.total_bounds
grid_size = 60000  # 60km in meters

# Generating the grid coordinates
cols = list(np.arange(xmin, xmax + grid_size, grid_size))
rows = list(np.arange(ymin, ymax + grid_size, grid_size))

polygons = []
for x in cols[:-1]:
    for y in rows[:-1]:
        polygons.append(Polygon([(x,y), (x+grid_size, y), (x+grid_size, y+grid_size), (x, y+grid_size)]))


grid = gpd.GeoDataFrame({'geometry': polygons}, crs="EPSG:32644")

fig, ax = plt.subplots(figsize=(10, 10))

# Plotting the NCR boundary
ncr_metric.plot(ax=ax, color='lightgrey', edgecolor='black')

# Overlaying the 60km grid boundaries in red
grid.boundary.plot(ax=ax, color='red', linewidth=1)

plt.title("Delhi-NCR with 60×60 km Uniform Grid (EPSG:32644)")
plt.xlabel("Easting (meters)")
plt.ylabel("Northing (meters)")
plt.show()

##b. Filter satellite images whose center coordinates fall inside the region.
##c. Report the total number of images before and after filtering.


In [ ]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1.Folder path
image_dir = '/content/drive/MyDrive/AI_Sustainability/Delhi/rgb'
all_files = [f for f in os.listdir(image_dir) if f.endswith('.png')]
before_count = len(all_files)

# 2. Extracting Lat/Lon from filenames and creating Points
data = []
for f in all_files:
    parts = f.replace('.png', '').split('_')
    lat, lon = float(parts[0]), float(parts[1])
    data.append({'filename': f, 'geometry': Point(lon, lat)})

# 3. Creating a GeoDataFrame of all image centers
gdf_images = gpd.GeoDataFrame(data, crs="EPSG:4326")

# 4. FILTER: To keep images within NCR region only
filtered_gdf = gdf_images[gdf_images.within(ncr.union_all())]
after_count = len(filtered_gdf)

# 5. Printing results
print(f"Total images before filtering: {before_count}")
print(f"Total images after filtering: {after_count}")

# Saving this for future reference
filtered_gdf.to_csv('/content/drive/MyDrive/AI_Sustainability/Delhi/filtered_images_1b.csv', index=False)

#Q2. Label Construction & Dataset Preparation

##a. For each image, extract the  128×128 corresponding land-cover patch from land_cover.tif using its center coordinate
##b. Assign the image label using the dominant (mode) land-cover class.
##c. Map ESA class codes to simplified land-use categories (e.g., Built-up, Vegetation, Water, Cropland, Others).
##d. Perform a 60/40 train-test split randomly and visualize class distribution


In [ ]:
!pip install rasterio geopandas shapely scikit-learn --quiet
import rasterio
from scipy import stats
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt

#1. Loading raster file
lc_path = '/content/drive/MyDrive/AI_Sustainability/Delhi/worldcover_bbox_delhi_ncr_2021.tif'
src = rasterio.open(lc_path)

#2. Loading the saved CSV file
filtered_gdf = gpd.read_file('/content/drive/MyDrive/AI_Sustainability/Delhi/filtered_images_1b.csv')

#3. Converting the 'geometry' column from string to shapely Point objects
filtered_gdf['geometry'] = filtered_gdf['geometry'].apply(wkt.loads)

# 4. Defining the mapping for simplified categories
# ESA WorldCover 2021 Codes: 10:Trees, 20:Shrub, 30:Grass, 40:Crop, 50:Built-up, 80:Water
esa_map = {
    10: 'Vegetation', 20: 'Vegetation', 30: 'Vegetation',
    40: 'Cropland',
    50: 'Built-up',
    80: 'Water'
}

def process_label(row):
    # 5. Extracting 128x128 patch and getting pixel coordinates from Lat/lon
    lon, lat = row['geometry'].x, row['geometry'].y
    py, px = src.index(lon, lat)

    # Defining a 128x128 window centered at the coordinate
    window = rasterio.windows.Window(px - 64, py - 64, 128, 128)

    patch = src.read(1, window=window)

    # 6. Assigning label using dominant (mode) class
    if patch.size > 0:
        mode_result = stats.mode(patch, axis=None, keepdims=True)
        dominant_code = int(mode_result.mode.item())

        # 7. Mapping to simplified or esa categories
        return esa_map.get(dominant_code, 'Others')
    else:
        return 'Others'

filtered_gdf['category'] = filtered_gdf.apply(process_label, axis=1)

from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Performing 60/40 train-test split
train_df, test_df = train_test_split(
    filtered_gdf,
    test_size=0.40,
    random_state=42,
    stratify=filtered_gdf['category']
)

class_counts = filtered_gdf['category'].value_counts()

# 2. Printing the numeric breakdown with names
print("--- Dataset Label Distribution ---")
print(class_counts)

# 3. Visualizing class distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=filtered_gdf, x='category', hue='category', palette='viridis', legend=False)
plt.title('Distribution of Land-Use Categories in Delhi-NCR')
plt.xlabel('Category')
plt.ylabel('Number of Images')


# Adding the exact count on top of each bar
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 5, str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

#Q3. Model Training & Supervised Evaluation

##1. Training a CNN Model

##2. Evaluation: Accuracy and F1-Score

##3. Confusion Matrix and Interpretation

####Initially Code 1 was plotted. But, it ignored the rare classes like "Water", as weights were not used in loss function which resulted F1-score for minority classes to be 0.00. Hence weights were introduced in the loss function in Code 2. Also, code 1 had a accuracy rate around 89% approximately which code 2 improved to 92% approximately.

####Code 1

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np
import random


seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

class DelhiDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
        # Creating a numerical mapping for the categories
        self.cat_to_idx = {'Vegetation': 0, 'Cropland': 1, 'Built-up': 2, 'Water': 3, 'Others': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        image = Image.open(img_name).convert('RGB')
        label = self.cat_to_idx[self.df.iloc[idx]['category']]

        if self.transform:
            image = self.transform(image)
        return image, label

# Image Transforms
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Creating Loaders
train_ds = DelhiDataset(train_df, '/content/drive/MyDrive/AI_Sustainability/Delhi/rgb', transform=data_transforms)
test_ds = DelhiDataset(test_df, '/content/drive/MyDrive/AI_Sustainability/Delhi/rgb', transform=data_transforms)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# Model Setup
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

import torch.optim as optim

# 1. Defining Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop
num_epochs = 10
print("Starting Training...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)
        loss = criterion(outputs, labels)


        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

print("Training Complete!")


# Evaluation: Accuracy and F1-Score
from sklearn.metrics import accuracy_score, f1_score

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1-Score (Weighted): {f1_score(all_labels, all_preds, average='weighted'):.4f}")


# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)
classes = ['Vegetation', 'Cropland', 'Built-up', 'Water', 'Others']

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Delhi Land Use Audit')
plt.show()


print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

####Code 2

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# 2. Dataset Class
class DelhiDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.cat_to_idx = {'Vegetation': 0, 'Cropland': 1, 'Built-up': 2, 'Water': 3, 'Others': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        image = Image.open(img_name).convert('RGB')
        label = self.cat_to_idx[self.df.iloc[idx]['category']]
        if self.transform:
            image = self.transform(image)
        return image, label

# 3. Data Preprocessing & Splitting
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Splitting train_df into Train (80%) and Val (20%)
train_sub_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=42
)

# 4. Data Loaders
image_path = '/content/drive/MyDrive/AI_Sustainability/Delhi/rgb'

train_ds = DelhiDataset(train_sub_df, image_path, transform=data_transforms)
val_ds = DelhiDataset(val_df, image_path, transform=data_transforms)
test_ds = DelhiDataset(test_df, image_path, transform=data_transforms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# 5. Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(model.fc.in_features, 5)
model.to(device)

# 6. Loss Function with Class Weights (Handling Imbalance)
class_counts_train_sub = train_sub_df['category'].value_counts().reindex(
    ['Vegetation', 'Cropland', 'Built-up', 'Water', 'Others'], fill_value=0
)
weights = 1.0 / (torch.tensor(class_counts_train_sub.values, dtype=torch.float32) + 1e-5)
weights = weights / weights.sum()
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 7. Training Loop with Validation
num_epochs = 15
print("Starting Training...")

for epoch in range(num_epochs):
    # Training Phase
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * images.size(0)

    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()

    avg_train_loss = running_train_loss / len(train_loader.dataset)
    avg_val_loss = running_val_loss / len(val_loader.dataset)
    val_acc = (val_correct / len(val_loader.dataset)) * 100

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print("Training Complete!")

# 8. Evaluation: Accuracy and F1-Score
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Results
print(f"\nFinal Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"Final F1-Score (Weighted): {f1_score(all_labels, all_preds, average='weighted'):.4f}")

# 9. Confusion Matrix
classes = ['Vegetation', 'Cropland', 'Built-up', 'Water', 'Others']
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Delhi Land Use Audit')
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

#Class-wise Interpretation:

---

#Vegetation:

Actual Vegetation
[215  41  45  0  0]

###Out of 301 vegetation samples:

215 correctly predicted as vegetation

41 misclassified as cropland

45 misclassified as built-up

###Metrics:

Precision: 0.77

Recall: 0.71

F1-score: 0.74

###Interpretation

Vegetation is sometimes confused with cropland and built-up areas.


#Cropland

Actual Cropland
[40 2112 38 0 0]

###Out of 2190 cropland samples:

2112 correctly predicted

40 predicted as vegetation

38 predicted as built-up

###Metrics:

Precision: 0.96

Recall: 0.96

F1-score: 0.96

###Interpretation

Cropland is the most accurately predicted class.

#Built-up

Actual Built-up
[23 40 648 0 0]

###Out of 711 built-up samples:

648 correctly classified

23 misclassified as vegetation

40 misclassified as cropland

###Metrics:

Precision: 0.89

Recall: 0.91

F1-score: 0.90

###Interpretation

Built-up areas are classified quite well, but some confusion occurs with cropland and vegetation.

#Water

Actual Water
[1 0 0 2 0]

###Out of 3 water samples:

2 correctly predicted

1 misclassified as vegetation

###Metrics:

Precision: 1.00

Recall: 0.67

F1-score: 0.80

###Interpretation

Water classification appears strong, but the sample size is extremely small, so the metrics are unreliable.

#Others

Actual Others
[1 0 0 0 0]

###Out of 1 sample:

0 correctly predicted

1 predicted as vegetation

###Metrics:

Precision: 0

Recall: 0

F1-score: 0

###Interpretation

The Others class has only 1 sample, so the model could not learn meaningful patterns.

##4. Visualization

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# 1. Combine all data for a full map (or just use test_df if you prefer)
full_df = pd.concat([train_df, test_df]).reset_index(drop=True)
full_ds = DelhiDataset(full_df, image_path, transform=data_transforms)
full_loader = DataLoader(full_ds, batch_size=32, shuffle=False)

# 2. Model Inference
model.eval()
all_predictions = []

print("Generating predictions for the map...")
with torch.no_grad():
    for images, _ in full_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_predictions.extend(preds.cpu().numpy())

# 3. Add predictions back to the dataframe
# Map the numbers back to category names
idx_to_cat = {0: 'Vegetation', 1: 'Cropland', 2: 'Built-up', 3: 'Water', 4: 'Others'}
full_df['predicted_category'] = [idx_to_cat[p] for p in all_predictions]

# Extract longitude and latitude from the geometry column
full_df['longitude'] = full_df['geometry'].apply(lambda p: p.x)
full_df['latitude'] = full_df['geometry'].apply(lambda p: p.y)

# 4. Create a GeoDataFrame
# Note: Ensure your 'full_df' has 'latitude' and 'longitude' columns as per the dataset
geometry = [Point(xy) for xy in zip(full_df['longitude'], full_df['latitude'])]
gdf_predictions = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")

# 5. Load the boundary (adjust path to your Q1/Q2 boundary file)
ncr_boundary = gpd.read_file('/content/drive/MyDrive/AI_Sustainability/Delhi/delhi_ncr_region.geojson')

# 6. Plotting
fig, ax = plt.subplots(figsize=(12, 10))
ncr_boundary.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.5)

# Define a color map for categories
color_map = {
    'Vegetation': 'green',
    'Cropland': '#fcd544',
    'Built-up': 'red',
    'Water': 'blue',
    'Others': 'purple'
}

for cat, color in color_map.items():
    subset = gdf_predictions[gdf_predictions['predicted_category'] == cat]
    subset.plot(ax=ax, color=color, label=cat, markersize=10, alpha=0.7)

plt.title("Predicted Land Use Pattern in Delhi-NCR", fontsize=15)
plt.legend(title="Land Use Classes", loc='upper right')
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()